# Esperimenti bottom-up: prompting e temperatura a soglie fisse

Il notebook prepara **5 configurazioni su tutti i 12 dataset: 60 run per ripetizione**. La soglia unica e fissa e **0.68 per tutti i dataset e tutte le configurazioni**, come testato e confermato dall'utente. Non e richiesto alcun Excel di tuning per avviare gli esperimenti.

| Configurazione | Esempi evaluator | Temperatura evaluator |
|---|---:|---:|
| reference (few-shot) | 4 | 0.3 |
| E1_0_examples (zero-shot) | 0 | 0.3 |
| E1_1_examples (one-shot) | 1 | 0.3 |
| E2_temperature_0.0 | 4 | 0 |
| E2_temperature_0.6 | 4 | 0.6 |

Il disegno OFAT varia separatamente prompting e temperatura. Il riferimento è condiviso: non permette di stimare interazioni fra fattori. One-shot usa il primo dei quattro esempi few-shot esistenti; gli esempi sono indipendenti dalle ground truth. Il confronto misura quindi anche l'effetto del contenuto degli esempi scelti.

La soglia 0.68 per HLG ricostruiti-originali, il modello embedding, i generatori e i critic top-down restano fissi. Le cosine possono cambiare quando cambiano i testi. Ogni run parte dalla stessa baseline top-down congelata del dataset. Ground truth usate solo nella valutazione esterna, separata dal tuning e dal feedback.

Le suite storiche conservano il proprio manifest; il loro precedente disegno a 10 configurazioni non va presentato come parte di questa campagna. Eseguire in un kernel dedicato, senza altre inferenze sullo stesso server LM Studio.

Questo notebook esegue gli esperimenti e salva gli output. Precision, Recall, F1, matching BERT e grafici comparativi si calcolano esclusivamente nel notebook 04.


Few-shot usa 4 esempi. Il confronto prompting usa temperatura 0.3; il confronto temperature usa few-shot a 0, 0.3 e 0.6. La condizione few-shot a 0.3 e condivisa e viene eseguita una sola volta. Per applicare il nuovo disegno, preparare una nuova suite.

Ogni evaluator riceve tutta la gerarchia. Una risposta incompleta o invalida viene scartata interamente e ripetuta solo per quel modello, fino a tre tentativi; i voti completi degli altri modelli restano validi. Il client verifica almeno 16384 token di contesto con le impostazioni predefinite; il budget di output degli evaluator resta 4096 token e quello Qwen e 8192. Prompt e payload sono compattati; questa policy richiede una nuova suite, separata dagli output storici.

La cosine e calcolata solo tra ogni HLG ricostruito e gli HLG originali, incluso il padre. I duplicati vengono valutati sui testi dei branch dello stesso attore, senza cosine fra ricostruzioni e senza soglia numerica sui duplicati.


In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Aprire il notebook dalla cartella del progetto o da notebook/")
for path in (PROJECT_ROOT, PROJECT_ROOT / "notebook"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))





## Funzioni degli esperimenti incorporate nel notebook

Le celle successive preparano le configurazioni e aggiornano un JSON per dataset. Il manifest incorpora baseline, codice di supporto, hash e provenienza. Le configurazioni e ripetizioni sono voci dello stesso documento, senza copie delle iterazioni o CSV operativi. Il notebook 04 legge anche gli archivi storici.


### Configurazioni e funzioni comuni


In [ ]:
from collections import Counter
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
import hashlib
from importlib.metadata import PackageNotFoundError, version
import json
from pathlib import Path
import platform
import re
import time
import traceback


@dataclass(frozen=True)
class ExperimentConfig:
    parent_threshold: float
    config_id: str = "reference"
    experiment: str = "reference"
    evaluator_few_shot_examples: int = 4
    evaluator_temperature: float = 0.3

    def __post_init__(self):
        from src.bottom_up.goal_similarity import similarity_configuration
        similarity_configuration(parent_threshold=self.parent_threshold)

    def cycle_options(self):
        return {key: value for key, value in asdict(self).items()
                if key not in {"config_id", "experiment"}}


SIMILARITY_THRESHOLD = 0.68


def fixed_threshold_selection():
    """Shared threshold tested and explicitly confirmed by the user."""
    from src.bottom_up.goal_similarity import similarity_configuration
    embedding = similarity_configuration(parent_threshold=SIMILARITY_THRESHOLD)
    return dict(similarity_threshold=SIMILARITY_THRESHOLD,
                parent_threshold=SIMILARITY_THRESHOLD,
                selection_source="user_tested_fixed",
                selection_note="Soglia 0.68 testata e confermata dall'utente per tutti i dataset e le configurazioni.",
                embedding_configuration=embedding)


def experiment_configurations(threshold_selection):
    """OFAT: vary prompting and temperature independently; thresholds stay fixed."""
    if threshold_selection['parent_threshold'] != SIMILARITY_THRESHOLD:
        raise ValueError("Tutte le configurazioni richiedono la soglia fissa 0.68")
    base = ExperimentConfig(parent_threshold=threshold_selection['parent_threshold'])
    return [base] + [
        replace(base, config_id=f"E1_{count}_examples", experiment="E1_prompting",
                evaluator_few_shot_examples=count) for count in (0, 1)
    ] + [
        replace(base, config_id=f"E2_temperature_{temperature:.1f}", experiment="E2_temperature",
                evaluator_temperature=temperature) for temperature in (0., .6)
    ]


def _json_bytes(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2).encode("utf-8")


def _digest(raw):
    return hashlib.sha256(raw).hexdigest()


def _write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(_json_bytes(value))
    temporary.replace(path)


def _read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def _slug(name):
    label = re.sub(r'[^\w.-]+', '_', name).strip('._')[:100]
    if not label:
        raise ValueError("Empty dataset identifier")
    return label + "_" + _digest(name.encode())[:8]


def runtime_configuration():
    from src import llm_clients as clients
    from src.bottom_up import global_goal_evaluator as evaluator
    from src.bottom_up import goal_cycle_orchestrator as cycle
    from src.bottom_up import goal_similarity as similarity
    from src.self_critique import refine_response as reflection
    packages = {}
    for name in ("openai", "pydantic", "numpy", "sentence-transformers", "torch", "transformers", "pandas", "scipy"):
        try:
            packages[name] = version(name)
        except PackageNotFoundError:
            packages[name] = None
    return {
        "policy": evaluator.PIPELINE_POLICY_VERSION,
        "generator_and_reconstructor": clients.QWEN_GENERATOR_MODEL,
        "evaluator_models": list(evaluator.BOTTOM_UP_EVALUATOR_MODELS),
        "repair_critic": clients.configured_critic_model(),
        "quorum": evaluator.EVALUATOR_VOTING_QUORUM,
        "embedding_model": similarity.EMBEDDING_MODEL,
        "embedding_revision": similarity.EMBEDDING_REVISION,
        "generation_mode": "ZERO_SHOT", "generation_evaluator_ablation": False,
        "repair_critic_examples": 4,
        "repair_quality_threshold": reflection.QUALITY_THRESHOLD,
        "repair_max_attempts": reflection.MAX_ATTEMPTS,
        "repair_policy": "original_top_down_start_at_hlg_then_llgs",
        "repair_acceptance": "hlg_critic_approval_then_llg_structural_validation_atomic_commit",
        "max_hlg_collection_repair_passes": cycle.MAX_HLG_COLLECTION_REPAIR_PASSES,
        "endpoint": clients.LM_STUDIO_BASE_URL,
        "context_length": clients.LM_STUDIO_EVALUATOR_CONTEXT_LENGTH,
        "evaluator_max_tokens": clients.BOTTOM_UP_CRITIC_MAX_TOKENS,
        "generator_max_tokens": clients.QWEN_GENERATOR_MAX_TOKENS,
        "vote_attempts": evaluator.BOTTOM_UP_EVALUATOR_VOTE_ATTEMPTS,
        "evaluator_retry_policy": "whole_global_response_per_model",
        "review_payload_format": "compact_matrix_v2_textual_duplicates",
        "rationale_max_words": 25, "critique_max_words": 60,
        "request_timeout_seconds": clients.REQUEST_TIMEOUT_SECONDS,
        "api_max_retries": clients.API_MAX_RETRIES,
        "python": platform.python_version(),
        "packages": packages,
        "seed_policy": "No seed forced; repetitions are independent calls, not guaranteed deterministic.",
        "temperature_support": "Requested temperature recorded in each generation metadata; no silent fallback.",
    }




### Preparazione, controlli e ciclo di esecuzione


In [ ]:
def notebook_helpers_source():
    notebook_path = PROJECT_ROOT / "notebook" / "03_bottom_up_experiments.ipynb"
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    return "\n\n".join(
        "".join(cell["source"]) for cell in notebook["cells"]
        if "experiment_helpers" in cell.get("metadata", {}).get("tags", [])
    )


def prepare_suite(project_root, baseline_files, suite_id, *, repetitions=1,
                  configurations=None, max_iterations=25, expected_datasets=12):
    """Freeze all baselines before the first model call; refuse changed resumes."""
    from src.bottom_up.global_goal_evaluator import evaluator_examples, SYSTEM_PROMPT
    from src.data_model import HighLevelGoals, LowLevelGoals
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", suite_id):
        raise ValueError("Use a simple suite_id without path separators")
    if type(repetitions) is not int or repetitions < 1 or not 1 <= max_iterations <= 25:
        raise ValueError("Require positive integer repetitions and 1..25 iterations")
    selection = fixed_threshold_selection()
    if expected_datasets != 12:
        raise ValueError("The declared experimental design requires all 12 datasets")
    root = Path(project_root).resolve()
    directory = root / "output" / "bottom_up_experiments" / suite_id
    configs = list(configurations if configurations is not None else experiment_configurations(selection))
    allowed = {c.config_id: c for c in experiment_configurations(selection)}
    if not configs or len({c.config_id for c in configs}) != len(configs):
        raise ValueError("Require distinct experiment configurations")
    if len(configs) != 5 or any(allowed.get(c.config_id) != c for c in configs):
        raise ValueError("Configurations must match the declared OFAT design")
    baselines, frozen = [], {}
    for path in baseline_files:
        path = Path(path).resolve()
        raw = path.read_bytes()
        data = json.loads(raw)
        HighLevelGoals.model_validate(data["structuredHighLevelGoals"])
        LowLevelGoals.model_validate(data["structuredLowLevelGoals"])
        if not data.get("name") or not data.get("description"):
            raise ValueError(f"Missing project name/description: {path}")
        slug = _slug(data["name"])
        if slug in frozen:
            raise ValueError(f"Duplicate dataset: {data['name']}")
        # Freeze source documents and initial hierarchies only, never references.
        frozen[slug] = raw
        baselines.append({"dataset": data["name"], "dataset_id": slug,
                          "source_file": str(path), "sha256": _digest(_json_bytes(data)), "source_sha256": _digest(raw),
                          "payload": data, "dataset_file": f"datasets/{slug}.json"})
    if len(baselines) != expected_datasets:
        raise ValueError(f"Expected {expected_datasets} datasets, found {len(baselines)}")
    code_files = sorted((root / "src").rglob("*.py"))
    manifest = {
        "schema_version": 2, "suite_id": suite_id, "storage_layout": "dataset_json_v2",
        "configurations": [asdict(c) for c in configs], "repetitions": repetitions,
        "max_iterations": max_iterations, "baselines": baselines,
        "planned_runs": len(configs) * len(baselines) * repetitions,
        "fixed_runtime": runtime_configuration(),
        "code_sha256": {str(p.relative_to(root)): _digest(p.read_bytes()) for p in code_files},
        "notebook_helpers_sha256": _digest(notebook_helpers_source().encode("utf-8")),
        "notebook_helpers_snapshot": notebook_helpers_source(),
        "examples_by_count": {str(n): evaluator_examples(n) for n in (0, 1, 4)},
        "evaluator_system_prompt": SYSTEM_PROMPT,
        "design": "OFAT prompting 0/1/4 examples; evaluator temperature 0/0.3/0.6; prompting at 0.3; fixed shared threshold 0.68 tested by the user.",
        "threshold_selection": selection,
        "ground_truth_usage": "Post-hoc evaluation only; never passed to the cycle.",
    }
    path = directory / "manifest.json"
    if path.exists():
        existing = _read(path)
        if {k: v for k, v in existing.items() if k not in {"created_at", "temperature_preflight"}} != manifest:
            raise ValueError("Suite inputs/code/settings changed. Choose a new suite_id.")
    else:
        if directory.exists() and any(directory.iterdir()):
            raise ValueError("Nonempty suite directory without manifest")
        _write_json(path, {**manifest, "created_at": datetime.now(timezone.utc).isoformat()})
    from src.bottom_up.result_storage import DatasetResults, write_summary
    for item in baselines:
        store = DatasetResults(directory / item["dataset_file"], item["payload"])
        for config in configs:
            for repetition in range(1, repetitions + 1):
                key = f"{config.config_id}/repeat_{repetition:03d}"
                store.data["runs"].setdefault(key, dict(status="not_started", experiment=asdict(config),
                    repetition=repetition, converged=None, final_decisions=[]))
        store.save()
    write_summary(directory, [directory / b["dataset_file"] for b in baselines],
                  planned_runs=manifest["planned_runs"])
    return directory


def _verify_suite(directory):
    manifest = _read(directory / "manifest.json")
    if manifest.get("storage_layout") != "dataset_json_v2":
        raise ValueError("Historical suite: evaluate it with notebook 04; prepare a new compact suite to run")
    root = directory.parents[2]
    if manifest["notebook_helpers_sha256"] != _digest(notebook_helpers_source().encode("utf-8")):
        raise ValueError("Notebook helpers changed; prepare a new suite")
    if manifest["fixed_runtime"] != runtime_configuration():
        raise ValueError("Runtime changed; prepare a new suite")
    for relative, checksum in manifest["code_sha256"].items():
        if _digest((root / relative).read_bytes()) != checksum:
            raise ValueError(f"Code changed: {relative}")
    for baseline in manifest["baselines"]:
        if _digest(_json_bytes(baseline["payload"])) != baseline["sha256"]:
            raise ValueError("Frozen baseline altered")
    return manifest


def _completed_result_error(record, payload):
    """A final result, including normal limit stops, must have a full trace."""
    from src.data_model import GlobalGoalCycleResult
    if record.get("status") != "completed" or record.get("error"):
        return "Run failed or interrupted before completion"
    try:
        cycle = GlobalGoalCycleResult.model_validate(payload["globalGoalCycle"])
    except (KeyError, ValueError, TypeError) as exc:
        return f"Invalid final result: {exc}"
    traces = cycle.iterations
    if not traces or len(traces) != cycle.completed_iterations:
        return "Missing completed iteration trace"
    if [t.iteration for t in traces] != list(range(1, len(traces) + 1)):
        return "Incomplete iteration sequence"
    if any(not t.checkpoint_complete for t in traces):
        return "Incomplete iteration checkpoint"
    last = traces[-1]
    if (last.high_level_goals != cycle.final_high_level_goals
            or last.low_level_goals != cycle.final_low_level_goals
            or payload.get("finalHighLevelGoals") != cycle.final_high_level_goals.model_dump(mode="json")
            or payload.get("finalLowLevelGoals") != cycle.final_low_level_goals.model_dump(mode="json")):
        return "Final hierarchy does not match the last completed iteration"
    if cycle.stop_reason.value not in {
            "ALL_BRANCHES_CONFIRMED", "MAX_ITERATIONS_REACHED", "NO_ACTIONS_REMAIN_WITH_WARNINGS"}:
        return "Unexpected stop reason"
    if cycle.converged != (cycle.stop_reason.value == "ALL_BRANCHES_CONFIRMED"):
        return "Inconsistent convergence status"
    if any("SIMILARITY_UNAVAILABLE" in w or "EVALUATOR_CONTEXT_LENGTH_EXCEEDED" in w for w in cycle.warnings):
        return "Run stopped because comparison or model context was unavailable"
    return None


def run_suite(directory, *, selected_config_ids=None, verbose=True, retry_failed=True, reuse_completed=True):
    """Update one dataset document; each configuration/repetition is a run entry."""
    from src import llm_clients
    from src.bottom_up.goal_cycle_orchestrator import run_global_goal_cycle
    from src.bottom_up.result_storage import DatasetResults, result_payload, write_summary
    from src.data_model import Actors, HighLevelGoals, LowLevelGoals
    from src.examples.shot_learning import ShotPromptingMode
    directory = Path(directory).resolve()
    manifest = _verify_suite(directory)
    available = {c["config_id"] for c in manifest["configurations"]}
    selected = available if selected_config_ids is None else set(selected_config_ids)
    if not selected <= available:
        raise ValueError(f"Unknown configurations: {selected - available}")
    files = [directory / b["dataset_file"] for b in manifest["baselines"]]
    def summarize():
        return write_summary(directory, files, planned_runs=manifest["planned_runs"])
    try:
        for config in manifest["configurations"]:
            if config["config_id"] not in selected:
                continue
            for repetition in range(1, manifest["repetitions"] + 1):
                key = f"{config['config_id']}/repeat_{repetition:03d}"
                for baseline in manifest["baselines"]:
                    data = baseline["payload"]
                    store = DatasetResults(directory / baseline["dataset_file"], data)
                    previous = store.data["runs"].get(key, {})
                    if previous.get("status") == "completed" and reuse_completed:
                        if _completed_result_error(previous, result_payload(store.data, previous)) is None:
                            print(f"[skip completed] {baseline['dataset']} / {key}")
                            continue
                    if previous.get("status") in {"failed", "interrupted"} and not retry_failed:
                        continue
                    store.start(key, configuration=config, repetition=repetition)
                    started = time.perf_counter()
                    print(f"{baseline['dataset']} / {key}", flush=True)
                    try:
                        llm_clients.reset_model_clients()
                        result = run_global_goal_cycle(
                            project_description=data["description"],
                            actors=Actors.model_validate(data["structuredActors"]) if data.get("structuredActors") else None,
                            initial_high_level_goals=HighLevelGoals.model_validate(data["structuredHighLevelGoals"]),
                            initial_low_level_goals=LowLevelGoals.model_validate(data["structuredLowLevelGoals"]),
                            mode=ShotPromptingMode.ZERO_SHOT, evaluator_ablation=False,
                            max_iterations=manifest["max_iterations"], verbose=verbose,
                            on_state=lambda cycle: store.update(key, cycle),
                            **ExperimentConfig(**config).cycle_options())
                        store.complete(key, result)
                    except BaseException as exc:
                        store.fail(key, f"{type(exc).__name__}: {exc}", interrupted=not isinstance(exc, Exception))
                        store.data["runs"][key]["error_traceback"] = traceback.format_exc()
                        if not isinstance(exc, Exception):
                            raise
                    finally:
                        store.data["runs"][key]["elapsed_seconds"] = time.perf_counter() - started
                        store.save()
                        llm_clients.reset_model_clients()
                        summarize()
    finally:
        summarize()
    return export_operational_tables(directory)


### Viste in memoria dei risultati JSON


In [4]:
def export_operational_tables(directory):
    """In-memory notebook views; all persisted results stay in dataset JSONs."""
    from src.bottom_up.result_storage import read_json
    manifest = _read(Path(directory) / "manifest.json")
    runs, iterations, decisions = [], [], []
    for baseline in manifest["baselines"]:
        document = read_json(Path(directory) / baseline["dataset_file"])
        for key, run in document["runs"].items():
            config = run["experiment"]
            identity = dict(dataset=baseline["dataset"], config_id=config["config_id"],
                            experiment=config["experiment"], repetition=run["repetition"])
            runs.append({**identity, **{k: run.get(k) for k in (
                "status", "converged", "stop_reason", "completed_iterations", "elapsed_seconds", "error")}})
            for trace in run.get("globalGoalCycle", {}).get("iterations", []):
                iterations.append({**identity, "iteration": trace["iteration"],
                                   "checkpoint_complete": trace.get("checkpoint_complete", True),
                                   "branches": len(trace.get("evaluations", {}))})
                decisions.extend({**identity, "iteration": trace["iteration"], **decision}
                                 for decision in trace.get("decision_summary", []))
    frame = pd.DataFrame(runs)
    summaries = frame.groupby(["config_id", "status"], dropna=False).size().rename("runs").reset_index()
    decision_frame = pd.DataFrame(decisions)
    action_counts = (decision_frame.groupby(["config_id", "final_decision", "applied_change"], dropna=False)
                     .size().rename("count").reset_index() if decisions else pd.DataFrame())
    return dict(runs=frame, iterations=pd.DataFrame(iterations), decisions=decision_frame,
                configurations=pd.DataFrame(manifest["configurations"]), operational_summary=summaries,
                action_counts=action_counts)


## Configurazione della suite


In [ ]:
# Per riprendere una suite in un nuovo kernel, impostare il suo ID esistente.
SUITE_ID = "ofat_fixed_thresholds_" + datetime.now(ZoneInfo("Europe/Rome")).strftime("%Y%m%d_%H%M%S")
REPETITIONS = 1  # Impostare per esempio 3 PRIMA di preparare la suite: 180 run.
MAX_ITERATIONS = 25
EXPECTED_DATASETS = 12
VERBOSE = True
REUSE_COMPLETED_RESULTS = True  # Salta risultati completi compatibili anche di suite precedenti.
# Esecuzione completa: tutte le configurazioni e tutti i dataset, senza filtri.

BASELINE_FILES = sorted((PROJECT_ROOT / "output" / "top_down").glob(
    "*_ZERO_SHOT_feedback_original_top_down_top_down.json"))
# Parametro fisso della reflection, separato dai few-shot degli evaluator globali.
import src.self_critique.refine_response as reflection
reflection.BOTTOM_UP_EVALUATOR_FEW_SHOT_EXAMPLE_COUNT = 4
# Soglia unica gia testata dall'utente: non occorre il workbook di tuning.
THRESHOLD_SELECTION = fixed_threshold_selection()
print("Soglia fissa HLG ricostruiti-originali:", THRESHOLD_SELECTION['similarity_threshold'])
CONFIGURATIONS = experiment_configurations(THRESHOLD_SELECTION)
if len(CONFIGURATIONS) != 5 or len(BASELINE_FILES) != EXPECTED_DATASETS:
    raise RuntimeError("Il ciclo completo richiede 5 configurazioni e 12 baseline")
display(pd.DataFrame([vars(c) for c in CONFIGURATIONS]))
print("Suite:", SUITE_ID)
print("Baseline:", len(BASELINE_FILES), "Run pianificate:", len(CONFIGURATIONS) * EXPECTED_DATASETS * REPETITIONS)
print(json.dumps(runtime_configuration(), indent=2, ensure_ascii=False))


## 1. Congelamento degli input e provenienza

Questa cella valida tutte le baseline e le incorpora nel manifest della suite, prima di qualsiasi chiamata ai modelli. I nomi dei dataset provengono dai file top-down: **le ground truth non vengono caricate durante l'inferenza**. Il manifest conserva hash SHA-256 degli input e del codice, configurazioni, esempi esatti, modelli e parametri operativi.

Una ripresa con stesso `SUITE_ID` richiede input, codice e impostazioni identici. Per cambiare ripetizioni o configurazione creare una nuova suite. Le cartelle storiche `top_down_bottom_up` e `bottom_up_iterations` restano separate dagli esperimenti.


In [ ]:
SUITE_DIR = prepare_suite(
    PROJECT_ROOT, BASELINE_FILES, SUITE_ID,
    repetitions=REPETITIONS, configurations=CONFIGURATIONS,
    max_iterations=MAX_ITERATIONS, expected_datasets=EXPECTED_DATASETS,
)
print("Dataset JSONs:", SUITE_DIR / "datasets")
print("Combined JSON:", SUITE_DIR / "results_summary.json")


## 2. Verifica delle temperature sul backend locale

Questa verifica effettua una breve richiesta strutturata per ogni modello e temperatura prevista, senza usare dataset o ground truth. Registra l'accettazione delle richieste e i metadati restituiti; l'accettazione non dimostra che il backend rispetti internamente il campionamento. Una temperatura 0 non garantisce determinismo. La prova è separata dalle run sperimentali; in caso di errore correggere la configurazione del server e rieseguire la cella.


In [ ]:
def verify_temperature_requests():
    import src.llm_clients as clients
    from src.bottom_up.global_goal_evaluator import BOTTOM_UP_EVALUATOR_MODELS
    schema = {"type": "object", "properties": {"ok": {"type": "boolean", "enum": [True]}},
              "required": ["ok"], "additionalProperties": False}
    rows = []
    try:
        for model in BOTTOM_UP_EVALUATOR_MODELS:
            with clients.loaded_local_evaluator_model(model):
                for temperature in sorted({config.evaluator_temperature for config in CONFIGURATIONS}):
                    row = {"model": model, "temperature": temperature}
                    try:
                        raw = clients.generate_evaluator_response(
                            'Return {"ok":true}.', "Return only the requested JSON object.",
                            model=model, temperature=temperature, response_schema=schema)
                        if json.loads(raw) != {"ok": True}:
                            raise ValueError("Unexpected structured response")
                        row.update(accepted=True, generation=getattr(raw, "metadata", {}))
                    except Exception as exc:
                        row.update(accepted=False, error=f"{type(exc).__name__}: {exc}")
                    rows.append(row)
    finally:
        clients.reset_model_clients()
        manifest = _read(SUITE_DIR / "manifest.json")
        manifest["temperature_preflight"] = rows
        _write_json(SUITE_DIR / "manifest.json", manifest)
    display(pd.DataFrame(rows))
    if not rows or not all(row["accepted"] for row in rows):
        raise RuntimeError("Preflight incompleto: controllare temperature_preflight nel manifest")
    return rows

# Richieste reali a LM Studio (12 brevi chiamate).
temperature_preflight = verify_temperature_requests()


## Esecuzione

Le run complete della stessa suite possono essere saltate; quelle fallite o interrotte ripartono dalla baseline e aggiornano la stessa voce JSON. Gli output storici restano valutabili nel notebook 04. Non vengono create copie delle iterazioni, file per tentativo o CSV di riepilogo.


In [ ]:
if not all(row["accepted"] for row in temperature_preflight):
    raise RuntimeError("Completare prima il preflight delle temperature")
operational_tables = run_suite(SUITE_DIR, selected_config_ids=None, verbose=VERBOSE,
                               retry_failed=True, reuse_completed=REUSE_COMPLETED_RESULTS)
runs_df = operational_tables["runs"]
assert len(runs_df) == len(CONFIGURATIONS) * EXPECTED_DATASETS * REPETITIONS
display(runs_df)
print("Tutti i risultati:", SUITE_DIR / "results_summary.json")


## Controllo degli output

Ogni dataset contiene tutte le configurazioni e ripetizioni nel proprio JSON. Ogni iterazione aggiorna quel file; `results_summary.json` raccoglie tutti i dataset, i voti e le decisioni, i goal finali e lo stato del sistema. Un nuovo tentativo sostituisce la voce della stessa configurazione/ripetizione, senza creare altre cartelle.


In [ ]:
operational_tables = export_operational_tables(SUITE_DIR)
display(operational_tables["runs"])
display(operational_tables["operational_summary"])
display(operational_tables["action_counts"])
print("Le tabelle sono solo viste in memoria; non vengono esportati CSV operativi.")


## Valutazione degli output

Aprire [04_evaluation_all_configurations.ipynb](04_evaluation_all_configurations.ipynb). Il notebook valuta tutte le suite disponibili, incluse quelle storiche, oltre alla baseline top-down e alla bottom-up ordinaria. Non esegue nuove generazioni.

Questo notebook salva un JSON per dataset, `results_summary.json` e il manifest tecnico. Le tabelle di analisi sono calcolate soltanto nella valutazione esterna.
